# Creation of eval and train dataset of prose

Creates `prosa_test_text.csv` and `prosa_train_text.csv` file with prose test and train datasets.

In [1]:
%pip install numpy pandas nltk

Note: you may need to restart the kernel to use updated packages.Requirement already satisfied: numpy in c:\hse\project_poetry_2\projectpoetryrl\.venv\lib\site-packages (2.2.6)




[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import numpy as np 
import pandas as pd 
import nltk
from nltk.tokenize import sent_tokenize
import os
import re

nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Марина\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Марина\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

To use this notebook download source dateset of Russian Literature from https://www.kaggle.com/datasets/d0rj3228/russian-literature as zip file. Then unzip it to './archive/'.

Functions for reading files and extracting chunks.

In [6]:
def read_first(filename, count_lines=None):
    lines = []
    with open(filename, 'r', encoding='utf-8') as file:
            for i, line in enumerate(file):
                if count_lines and i >= count_lines:
                    break
                lines.append(line.strip())  # Удаляем лишние символы новой строки
    return ' '.join(lines)

def split_text_into_chunks(paragraphs, min_length=210, max_length=310):
    sentences = sent_tokenize(paragraphs)
    chunks = []
    for sentence in sentences:
        sentence = sentence.strip()
        if len(sentence) <= max_length and len(sentence) >= min_length:
            chunks.append(sentence)
    return chunks

Extracting chunks

In [7]:
chunks = {}
chunks_long = {}
for dirname, _, filenames in os.walk('./archive/prose/'):
    chunks[dirname] = []
    chunks_long[dirname] = []
    for filename in filenames:
        if filename.endswith('.txt'):
            paragraphs = read_first(os.path.join(dirname, filename))
            chunks[dirname].extend(split_text_into_chunks(paragraphs, min_length=100, max_length=140))

In [8]:
all_chunks = []
for k, v in chunks.items():
    all_chunks.extend(v)

In [9]:
len(all_chunks)

51777

Examples of chunks and their lens.

In [26]:
# Фильтруем английские буквы
all_cleaned_chunks = [chunk for chunk in all_chunks if not re.search(r'[A-Za-z]', chunk)]

# Фильтруем спецсимволы (оставляем русские буквы, цифры, пробел и знаки препинания)
allowed_chars = re.compile(r'^[А-Яа-яЁё0-9 ,.!?—:;«»"\(\)\-–…]+$')
all_cleaned_chunks = [chunk for chunk in all_cleaned_chunks if allowed_chars.match(chunk.strip())]

BAD_WORDS = {"сей", "оне", "ибо", "аз", "се"}

def no_archaic(s):
    return not any(w in s.lower().split() for w in BAD_WORDS)

def is_meta(s):
    s_low = s.lower()
    return (
        s_low.startswith("глава") or
        s_low.startswith("часть") or
        s_low.startswith("том")
    )

# Фильтруем «обрезанные диалоги»
def looks_like_sentence(s):
    return s.strip()[-1] in ".!?"

all_cleaned_chunks = [chunk for chunk in all_cleaned_chunks if (looks_like_sentence(chunk) and no_archaic(chunk) and not is_meta(chunk))]

# Проверяем на дубли
all_cleaned_chunks = list(dict.fromkeys(all_cleaned_chunks))  # сохраняем порядок, убираем дубли

print(f"Длина датасета: {len(all_cleaned_chunks)}")

Длина датасета: 43357


In [27]:
for chunk in all_cleaned_chunks[:10]:
    print(chunk, len(chunk))

Различаясь так сильно, они часто не ладят друг с другом и тем взаимно истребляют свои бессмертные души. 103
Так уж и знаешь, что проснешься завтра, а уж вся Россия ликует: у нас такой-то новый министр народного просвещения! 115
Председатель принимал вечером, и я возвращался от него к себе домой вечером, в час страшно поздний: в шесть часов вечера. 121
Капитан броненосца наклонился к моему уху и прошептал конфиденциально: - Как вы думаете, немцы придут? 102
- Ну вот, как, например, в южной Франции в 1794 году или в Париже после Коммуны, во время белого террора. 105
- Но ведь то были французы, - сказал он, - а ведь это - черт знает кто... В эту минуту у калитки ворот раздался робкий стук. 124
Ее осмотрели с ног до головы, и она бросилась в глухой мрак своего подъезда сквозь строй смелых и хорошо вооруженных граждан. 125
Проходя на днях по нашей улице, я остановился перед окном нового магазина и стал рассматривать бумагу, вставочки, папиросы и спички. 132
Уже из уст в уста переходило его 

Saving as a csv file and adding meter and rhyme scheme.

In [28]:
test_size = 1000
val_size = 100
np.random.seed(42)
np.random.shuffle(all_cleaned_chunks)
test_texts = all_cleaned_chunks[:test_size]
val_texts = all_cleaned_chunks[test_size:val_size + test_size]
train_texts = all_cleaned_chunks[val_size + test_size:]

In [29]:
def build_df(texts):
    n = len(texts)

    df = pd.DataFrame({'input': texts})

    # схемы рифмовки
    schemes = ['AABB', 'ABAB', 'ABBA']
    df['rhyme_scheme'] = [schemes[i % 3] for i in range(n)]

    # метры
    meters = ['choreios', 'iambos']
    df['meter'] = [meters[i % 2] for i in range(n)]

    return df

In [30]:
df_train = build_df(train_texts)
df_train.to_csv('prosa_train_text.csv', index=False)
df_train

,input,rhyme_scheme,meter
0,"Выехав за ворота, я рысью объехал чуть не на в...",AABB,choreios
1,"Привратник вызвал молодую рабыню, которая пров...",ABAB,iambos
2,— Побойся бога; как тебе пускаться в дорогу в ...,ABBA,choreios
3,"Толстой использовал этот, эпизод, характеризую...",AABB,iambos
4,Солдат с повязкой садится на койку к Гусеву и ...,ABAB,choreios
...,...,...,...
42252,"Наконец, они объявили мне, что я становлюсь ...",AABB,choreios
42253,"Он всё налгал, что говорил нашим о доносе: ник...",ABAB,iambos
42254,Николай в два слова купил за шесть тысяч семна...,ABBA,choreios
42255,"В театр я вошел с Гесперией рядом и видел, что...",AABB,iambos


In [31]:
df_test = build_df(test_texts)
df_test.to_csv('prosa_test_text.csv', index=False)

In [32]:
df_val = build_df(val_texts)
df_val.to_csv('prosa_val_text.csv', index=False)